# Atividade Spark

Considerando o dataset detalhado a seguir, extraia o conjunto de informações solicitadas.

### Dataset dados de COVID no Brasil

- Dados relativos a pacientes que realizaram exames de Covid19 no Brasil ate Marco 2020
- ~1.6GB
- 4.4M de instâncias


|  # | Nome do campo            | Descrição                                     |
| -: | ------------------------ | --------------------------------------------- |
|  0 | id                       | Identificador                                 |
|  1 | dataNotificacao          | Data da notificação                           |
|  2 | dataInicioSintomas       | Data do início dos sintomas                   |
|  3 | dataNascimento           | Data de nascimento                            |
|  4 | sintomas                 | Sintomas do paciente                          |
|  5 | profissionalSaude        | Relacionado a profissional de saúde           |
|  6 | cbo                      | Ocupação                                      |
|  7 | condicoes                | Condições do paciente                         |
|  8 | estadoTeste              | Estado do teste                               |
|  9 | dataTeste                | Data do teste                                 |
| 10 | tipoTeste                | Tipo de teste realizado                       |
| 11 | resultadoTeste           | Resultado do teste                            |
| 12 | paisOrigem               | País de origem do paciente                    |
| 13 | sexo                     | Sexo do paciente                              |
| 14 | bairro                   | Bairro do paciente                            |
| 15 | estado                   | Estado do paciente                            |
| 16 | estadoIBGE               | Estado do paciente — IBGE                     |
| 17 | municipio                | Município do paciente                         |
| 18 | municipioIBGE            | Município do paciente — IBGE                  |
| 19 | cep                      | CEP                                           |
| 20 | origem                   | Origem do paciente                            |
| 21 | cnes                     | Código da unidade de saúde                    |
| 22 | estadoNotificacao        | Estado da notificação                         |
| 23 | estadoNotificacaoIBGE    | Estado da notificação — IBGE                  |
| 24 | municipioNotificacao     | Município da notificação                      |
| 25 | municipioNotificacaoIBGE | Município da notificação — IBGE               |
| 26 | numeroNotificacao        | Número da notificação                         |
| 27 | excluido                 | ID de exclusão                                |
| 28 | validado                 | Local de validação                            |
| 29 | idade                    | Idade do paciente                             |
| 30 | dataEncerramento         | Data do encerramento da avaliação do paciente |
| 31 | evolucaoCaso             | Evolução do caso do paciente                  |
| 32 | classificacaoFinal       | Avaliação final do caso                       |


Informações a serem extraídas:

1. Quantidade de pacientes positivos para corona virus (resultadoTeste)
2. Quantidade de pacientes do sexo feminino positivos para corona virus (resultadoTeste)
3. Quantidade de pacientes de acordo com o sexo e o resultado do teste (resultadoTeste)
4. Sintomas mais comuns para casos positivos para corona virus
5. Sintomas mais comuns para casos não positivos para corona virus
6. Quantidade de casos positivos no Paraná
7. Municipio do Paraná com a maior quantidade de óbitos (evolucaoCaso)
8. Quantidade de municipios no Paraná com casos positivos
9. Quantidade de municipios no Paraná sem casos positivos
10. Estado com a maior taxa de falecimento de mulheres
11. Menor idade de mulher positiva para covid
12. Maior idade de mulher positiva para covid
13. Quantidade de casos positivos, em granularidade diária, ao longo do tempo
14. Quantidade de casos positivos, em granularidade semanal, ao longo do tempo
15. Quantidade de pacientes que faleceram (evolucaoCaso) separados pela idade
16. Taxa de óbito (evolucaoCaso) por idade
17. Idade média das mulheres positivas para covid
18. Município do Paraná com a maior quantidade de mulheres positivos para covid
19. Dia da semana com a maior quantidade de testes realizados
20. Dia da semana com maior quantidade de pacientes positivos para corona virus
21. Municipio, com mais de 500 testes, com a maior taxa de exames que não foram positivos para COVID (casos que foram testados porém não tiveram o resultadoTeste como Positivo)

**Dicas:**
- *Crie uma célula (Insert -> Insert Cell Below) para cada informação solicitada*

In [1]:
#instala pyspark (execução local)
# O notebook é autossuficiente: instala as bibliotecas que faltarem, prepara um Java compatível
# com o Spark e, na célula de download, baixa a base para a mesma pasta do notebook.
import os, sys, glob, re, shutil, subprocess, importlib, importlib.util, importlib.metadata

def instala(*pacotes):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *pacotes])
    importlib.invalidate_caches()

# PySpark 3.5.9, a versão do enunciado. Ele roda com Python 3.9 a 3.12 (recomendado: 3.11);
# no Python 3.14 a serialização de funções do Spark 3.5 não funciona.
if sys.version_info >= (3, 14):
    raise RuntimeError('Use um kernel com Python 3.11 (o PySpark 3.5.9 não funciona no Python 3.14).')
PYSPARK_VERSAO = '3.5.9'
try:
    pyspark_instalado = importlib.metadata.version('pyspark')
except importlib.metadata.PackageNotFoundError:
    pyspark_instalado = None
if pyspark_instalado != PYSPARK_VERSAO:
    if 'pyspark' in sys.modules:
        raise RuntimeError('Outra versão do PySpark já foi carregada: reinicie o kernel e execute novamente a partir desta célula.')
    instala(f'pyspark=={PYSPARK_VERSAO}')

for modulo, pacote in [('gdown', 'gdown')]:
    if importlib.util.find_spec(modulo) is None:
        instala(pacote)

# O Spark 3.5 precisa de Java 8, 11, 17 ou 21. Se não houver um desses na máquina,
# um JDK 17 portátil é baixado uma única vez para ~/.jdk (não altera o sistema).
def versao_java(executavel):
    try:
        saida = subprocess.run([executavel, '-version'], capture_output=True, text=True).stderr
        maior, menor = re.search(r'version "(\d+)(?:\.(\d+))?', saida).groups()
        return int(menor) if maior == '1' else int(maior)
    except Exception:
        return None

def java_disponivel():
    if os.environ.get('JAVA_HOME'):
        exe = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java')
        if shutil.which(exe) or os.path.exists(exe) or os.path.exists(exe + '.exe'):
            return exe
    return shutil.which('java')

if versao_java(java_disponivel()) not in (8, 11, 17, 21):
    if importlib.util.find_spec('jdk') is None:
        instala('install-jdk')
    import jdk
    pasta_jdk = os.path.join(os.path.expanduser('~'), '.jdk')
    instalados = sorted(glob.glob(os.path.join(pasta_jdk, 'jdk-17*')))
    java_home = instalados[-1] if instalados else jdk.install('17', path=pasta_jdk)
    if os.path.isdir(os.path.join(java_home, 'Contents', 'Home')):   # macOS
        java_home = os.path.join(java_home, 'Contents', 'Home')
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = os.path.join(java_home, 'bin') + os.pathsep + os.environ['PATH']

# os processos Python do Spark usam o mesmo Python deste notebook (necessário no Windows)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print('Python', sys.version.split()[0], '| Java', versao_java(java_disponivel()))

Python 3.11.18 | Java 21


In [2]:
# abre sessão no spark
import pyspark
conf = pyspark.SparkConf()
conf.setMaster('local[*]')
conf.set('spark.driver.memory', '4g')
conf.set('spark.driver.host', 'localhost')
conf.set('spark.sql.session.timeZone', 'UTC')
conf.set('spark.sql.ansi.enabled', 'false')
# no Windows, o worker Python cai após 15 s ocioso (TimeoutError); 1 hora evita isso
conf.set('spark.python.authenticate.socketTimeout', '1h')
sc = pyspark.SparkContext.getOrCreate(conf)
sc.setLogLevel('ERROR')
print('Spark', sc.version, '| paralelismo padrão:', sc.defaultParallelism)

Spark 3.5.9 | paralelismo padrão: 2


In [3]:
#Efetua download do arquivo covid
# baixa a base (cópia no Google Drive do aluno) para a pasta do notebook, se ainda não estiver lá
from pathlib import Path
import gdown

ARQUIVO = Path('covid_sample.csv')
if not ARQUIVO.exists():
    gdown.download(id='1gWzcibwqntoADi6-x4BUSOXrKqhNp192', output=str(ARQUIVO), quiet=False)
# file:///E:/pasta com espaços/arquivo.csv (sem codificar espaços e acentos, que o Hadoop não decodifica)
CAMINHO = 'file:///' + ARQUIVO.resolve().as_posix().lstrip('/')
print(ARQUIVO.name, f'{ARQUIVO.stat().st_size / 1e6:.0f} MB')

covid_sample.csv 370 MB


In [4]:
#carrega arquivo da COVID em um rdd
rdd = sc.textFile(CAMINHO)
print('Partições:', rdd.getNumPartitions())

Partições: 11


## Preparação dos dados

Toda a atividade é resolvida com a **API de RDD** (transformações `map`, `filter`, `flatMap`, `reduceByKey`, `sortBy`, `distinct`, `subtract`, `join` e ações `count`, `take`, `collect`, `reduce`), como nos slides de Apache Spark.

O arquivo é texto puro separado por `;`, com cabeçalho na primeira linha. Os passos abaixo:

1. guardam o cabeçalho e criam um dicionário `COL` com o índice de cada campo;
2. removem o cabeçalho e quebram cada linha em uma lista de 33 campos;
3. tratam como ausente (`None`) os marcadores `null`, `NULL`, `undefined` e `""` usados pela base;
4. fazem `cache()` do RDD resultante, porque ele é reutilizado em todas as 21 consultas.

In [5]:
cabecalho = rdd.first()
COL = {nome: i for i, nome in enumerate(cabecalho.split(';'))}
print(len(COL), 'campos')
print(COL)

33 campos
{'id': 0, 'dataNotificacao': 1, 'dataInicioSintomas': 2, 'dataNascimento': 3, 'sintomas': 4, 'profissionalSaude': 5, 'cbo': 6, 'condicoes': 7, 'estadoTeste': 8, 'dataTeste': 9, 'tipoTeste': 10, 'resultadoTeste': 11, 'paisOrigem': 12, 'sexo': 13, 'bairro': 14, 'estado': 15, 'estadoIBGE': 16, 'municipio': 17, 'municipioIBGE': 18, 'cep': 19, 'origem': 20, 'cnes': 21, 'estadoNotificacao': 22, 'estadoNotificacaoIBGE': 23, 'municipioNotificacao': 24, 'municipioNotificacaoIBGE': 25, 'numeroNotificacao': 26, 'excluido': 27, 'validado': 28, 'idade': 29, 'dataEncerramento': 30, 'evolucaoCaso': 31, 'classificacaoFinal': 32}


In [6]:
AUSENTES = {'', 'null', 'NULL', 'undefined'}

def limpa(valor):
    valor = valor.strip().strip('"').strip()
    return None if valor in AUSENTES else valor

def parse(linha):
    return [limpa(v) for v in linha.split(';')]

dados = rdd.filter(lambda linha: linha != cabecalho)\
           .map(parse)\
           .filter(lambda campos: len(campos) == len(COL))\
           .cache()

total = dados.count()
print('Registros (pacientes):', total)

Registros (pacientes): 1000000


In [7]:
# primeiro registro já limpo, exibido campo a campo
primeiro = dados.first()
for nome, i in COL.items():
    print(f'{i:2d} {nome:25s} {primeiro[i]}')

 0 id                        mPuc2Qohsn
 1 dataNotificacao           2020-06-29T13:37:08.188Z
 2 dataInicioSintomas        2020-06-26T03:00:00.000Z
 3 dataNascimento            1984-04-01T03:00:00.000Z
 4 sintomas                  Tosse, Febre, Outros
 5 profissionalSaude         Não
 6 cbo                       None
 7 condicoes                 None
 8 estadoTeste               Coletado
 9 dataTeste                 2020-06-29T03:00:00.000Z
10 tipoTeste                 None
11 resultadoTeste            None
12 paisOrigem                None
13 sexo                      Masculino
14 bairro                    Pq. Sta. Cecilia
15 estado                    SÃO PAULO
16 estadoIBGE                35
17 municipio                 São Paulo
18 municipioIBGE             3550308
19 cep                       04862
20 origem                    None
21 cnes                      3992101
22 estadoNotificacao         SÃO PAULO
23 estadoNotificacaoIBGE     35
24 municipioNotificacao      São Paulo
25 mu

In [8]:
import datetime

def campo(r, nome):
    return r[COL[nome]]

def positivo(r):
    return campo(r, 'resultadoTeste') == 'Positivo'

def mulher(r):
    return campo(r, 'sexo') == 'Feminino'

def obito(r):
    return campo(r, 'evolucaoCaso') == 'Óbito'

def parana(r):
    return campo(r, 'estado') == 'PARANÁ'

def idade(r):
    """Idade como inteiro; None quando ausente ou não numérica."""
    try:
        return int(campo(r, 'idade'))
    except (TypeError, ValueError):
        return None

def data(r, nome='dataTeste'):
    """Converte '2020-06-29T03:00:00.000Z' em datetime.date (só a parte da data)."""
    v = campo(r, nome)
    try:
        return datetime.date.fromisoformat(v[:10])
    except (TypeError, ValueError):
        return None

# grafias diferentes do mesmo sintoma encontradas na base
CANON = {'dor de garganta': 'Dor de Garganta', 'dispneia': 'Dispneia', 'dispnéia': 'Dispneia',
         'dispineia': 'Dispneia', 'dificuldade de respirar': 'Dificuldade de Respirar',
         'tosse': 'Tosse', 'febre': 'Febre', 'outros': 'Outros',
         'outros: paciente assintomático': 'Assintomático', 'paciente assintomático': 'Assintomático', 'diarreia': 'Diarreia', 'diarréia': 'Diarreia'}

import re
# alguns registros têm sintomas colados sem vírgula ('TosseDificuldade De Respirar')
COLADOS = re.compile(r'(tosse|febre|dispneia|outros|dor de garganta)(dificuldade|dor|febre|tosse|dispneia|outros)', re.I)

def sintomas(r):
    """Quebra o campo de sintomas ('Tosse, Febre, Outros') em uma lista de sintomas padronizados."""
    s = campo(r, 'sintomas')
    if s is None:
        return []
    s = COLADOS.sub(r'\1, \2', s)
    itens = [x.strip() for x in s.split(',') if x.strip()]
    return [CANON.get(x.lower(), x) for x in itens]

def municipio(r):
    """Nome do município em minúsculas: 'Foz Do Iguaçu' e 'Foz do Iguaçu' viram a mesma chave."""
    m = campo(r, 'municipio')
    return m.lower() if m else None

DIAS = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

**Definições adotadas** (valem para todas as respostas):

- **paciente positivo**: `resultadoTeste == 'Positivo'`; **não positivo**: qualquer outro valor, inclusive resultado ausente;
- **óbito**: `evolucaoCaso == 'Óbito'`;
- **Paraná**: `estado == 'PARANÁ'` (estado de residência do paciente);
- **data do teste**: parte da data de `dataTeste`; o dia da semana é obtido dela;
- **município**: comparado sem diferenciar maiúsculas e minúsculas, porque a base grava o mesmo município de duas formas (`Foz Do Iguaçu` e `Foz do Iguaçu`); o nome exibido é a grafia mais frequente;
- **sintomas**: o campo é uma lista separada por vírgulas; grafias diferentes do mesmo sintoma (`Dor De Garganta`, `Dispnéia`, `Dispineia`) são unificadas, `Outros: Paciente assintomático` vira `Assintomático` e sintomas colados sem vírgula (`TosseDificuldade De Respirar`) são separados;
- cada linha é contada como um paciente. O `id` tem 986.648 valores distintos em 1.000.000 de linhas (repetições do mesmo identificador em notificações diferentes), mas não há outro campo que identifique a pessoa, então a contagem é por registro.

In [9]:
# valores existentes nos campos categóricos usados nas consultas
ids_distintos = dados.map(lambda r: campo(r, 'id')).distinct().count()
print('ids distintos:', ids_distintos, '| registros:', total)
for nome in ['resultadoTeste', 'sexo', 'evolucaoCaso', 'estado']:
    contagem = dados.map(lambda r, n=nome: (campo(r, n), 1)).reduceByKey(lambda a, b: a + b)\
                    .sortBy(lambda kv: -kv[1]).collect()
    print(f'\n{nome}:')
    for k, v in contagem:
        print(f'   {str(k):28s} {v:>9,d}'.replace(',', '.'))

# nome de exibição de cada município: a grafia mais frequente na base
NOMES = dict(dados.map(lambda r: ((municipio(r), campo(r, 'municipio')), 1))
                  .filter(lambda kv: kv[0][0] is not None)
                  .reduceByKey(lambda a, b: a + b)
                  .map(lambda kv: (kv[0][0], (kv[1], kv[0][1])))
                  .reduceByKey(max)
                  .mapValues(lambda v: v[1])
                  .collect())
print('\nMunicípios distintos (sem diferenciar maiúsculas):', len(NOMES))

ids distintos: 986648 | registros: 1000000



resultadoTeste:
   Negativo                       576.383
   Positivo                       277.848
   None                           145.769



sexo:
   Feminino                       524.509
   Masculino                      451.023
   Indefinido                      24.368
   None                               100



evolucaoCaso:
   None                           656.249
   Cura                           231.188
   Ignorado                        40.049
   Cancelado                       36.856
   Em tratamento domiciliar        32.357
   Óbito                            2.119
   Internado                          967
   Internado em UTI                   215



estado:
   SÃO PAULO                      471.476
   MINAS GERAIS                   132.453
   BAHIA                          126.347
   RIO GRANDE DO SUL               99.278
   RIO DE JANEIRO                  94.453
   SANTA CATARINA                  66.560
   PARANÁ                           9.433



Municípios distintos (sem diferenciar maiúsculas): 3181


In [10]:
# 1. Quantidade de pacientes positivos para corona virus (resultadoTeste)
positivos = dados.filter(positivo).cache()
qt_positivos = positivos.count()
print('Pacientes positivos:', qt_positivos)

Pacientes positivos: 277848


**Resposta.** **277.848 pacientes positivos** para coronavírus (27,8 % dos 1.000.000 de registros).

In [11]:
# 2. Quantidade de pacientes do sexo feminino positivos para corona virus (resultadoTeste)
qt_fem_pos = positivos.filter(mulher).count()
print('Pacientes do sexo feminino positivos:', qt_fem_pos)

Pacientes do sexo feminino positivos: 142680


**Resposta.** **142.680 pacientes do sexo feminino positivos** (51,4 % dos positivos).

In [12]:
# 3. Quantidade de pacientes de acordo com o sexo e o resultado do teste (resultadoTeste)
sexo_resultado = dados.map(lambda r: ((campo(r, 'sexo'), campo(r, 'resultadoTeste')), 1))\
                      .reduceByKey(lambda a, b: a + b)\
                      .sortBy(lambda kv: (str(kv[0][0]), -kv[1]))\
                      .collect()
print(f"{'sexo':12s} {'resultadoTeste':15s} {'pacientes':>10s}")
for (sexo, resultado), qt in sexo_resultado:
    print(f'{str(sexo):12s} {str(resultado):15s} {qt:>10d}')

sexo         resultadoTeste   pacientes
Feminino     Negativo            298008
Feminino     Positivo            142680
Feminino     None                 83821
Indefinido   Negativo             20032
Indefinido   Positivo              4336
Masculino    Negativo            258333
Masculino    Positivo            130829
Masculino    None                 61861
None         None                    87
None         Negativo                10
None         Positivo                 3


**Resposta.** Pacientes por sexo e resultado do teste:

| Sexo | Positivo | Negativo | Sem resultado |
|---|---:|---:|---:|
| Feminino | 142.680 | 298.008 | 83.821 |
| Masculino | 130.829 | 258.333 | 61.861 |
| Indefinido | 4.336 | 20.032 | 0 |
| Sem informação | 3 | 10 | 87 |

A proporção de positivos entre os testados com resultado é parecida nos dois sexos (32,4 % nas mulheres e 33,6 % nos homens).

In [13]:
# 4. Sintomas mais comuns para casos positivos para corona virus
sint_pos = positivos.flatMap(sintomas)\
                    .map(lambda s: (s, 1))\
                    .reduceByKey(lambda a, b: a + b)\
                    .sortBy(lambda kv: -kv[1])
top_sint_pos = sint_pos.take(10)
for s, qt in top_sint_pos:
    print(f'{s:35s} {qt:>8d}  ({qt / qt_positivos:6.1%} dos positivos)')

Outros                                190532  ( 68.6% dos positivos)
Tosse                                 130130  ( 46.8% dos positivos)
Febre                                 104127  ( 37.5% dos positivos)
Dor de Garganta                        65332  ( 23.5% dos positivos)
Dispneia                               41840  ( 15.1% dos positivos)
Assintomático                           5550  (  2.0% dos positivos)
Dificuldade de Respirar                   15  (  0.0% dos positivos)
Diarreia                                   2  (  0.0% dos positivos)
perda de olfato e paladar                  1  (  0.0% dos positivos)
calafrio                                   1  (  0.0% dos positivos)


**Resposta.** Entre os positivos, os sintomas mais comuns são **Outros (68,6 %), Tosse (46,8 %), Febre (37,5 %), Dor de Garganta (23,5 %) e Dispneia (15,1 %)**, seguidos de Assintomático (2,0 %). O campo tem praticamente só essas seis categorias padronizadas; os demais itens do top 10 (Dificuldade de Respirar, Diarreia, perda de olfato e paladar, calafrio) são texto livre com poucas ocorrências. Um paciente pode ter vários sintomas, por isso as porcentagens somam mais de 100 %.

In [14]:
# 5. Sintomas mais comuns para casos não positivos para corona virus
nao_positivos = dados.filter(lambda r: not positivo(r))
qt_nao_pos = nao_positivos.count()
top_sint_nao = nao_positivos.flatMap(sintomas)\
                            .map(lambda s: (s, 1))\
                            .reduceByKey(lambda a, b: a + b)\
                            .sortBy(lambda kv: -kv[1])\
                            .take(10)
print('Pacientes não positivos:', qt_nao_pos)
for s, qt in top_sint_nao:
    print(f'{s:35s} {qt:>8d}  ({qt / qt_nao_pos:6.1%} dos não positivos)')

Pacientes não positivos: 722152
Outros                                472396  ( 65.4% dos não positivos)
Tosse                                 254871  ( 35.3% dos não positivos)
Febre                                 170278  ( 23.6% dos não positivos)
Dor de Garganta                       164839  ( 22.8% dos não positivos)
Dispneia                               92212  ( 12.8% dos não positivos)
Assintomático                          59079  (  8.2% dos não positivos)
Dificuldade de Respirar                 1210  (  0.2% dos não positivos)
Diarreia                                   2  (  0.0% dos não positivos)
dor no peito                               1  (  0.0% dos não positivos)
coriza                                     1  (  0.0% dos não positivos)


**Resposta.** Entre os 722.152 não positivos, a ordem dos cinco primeiros é a mesma: **Outros (65,4 %), Tosse (35,3 %), Febre (23,6 %), Dor de Garganta (22,8 %) e Dispneia (12,8 %)**, seguidos de Assintomático (8,2 %). A diferença está na intensidade: febre e tosse são bem mais frequentes entre os positivos (37,5 % contra 23,6 % e 46,8 % contra 35,3 %), e "assintomático" é quatro vezes mais frequente entre os não positivos.

In [15]:
# 6. Quantidade de casos positivos no Paraná
positivos_pr = positivos.filter(parana).cache()
qt_pos_pr = positivos_pr.count()
print('Casos positivos no Paraná:', qt_pos_pr)

Casos positivos no Paraná: 1419


**Resposta.** **1.419 casos positivos no Paraná** (pacientes com `estado == 'PARANÁ'`).

In [16]:
# 7. Municipio do Paraná com a maior quantidade de óbitos (evolucaoCaso)
obitos_pr = dados.filter(lambda r: parana(r) and obito(r))\
                 .map(lambda r: (municipio(r), 1))\
                 .reduceByKey(lambda a, b: a + b)\
                 .sortBy(lambda kv: -kv[1])
print('Total de óbitos no Paraná:', obitos_pr.map(lambda kv: kv[1]).sum())
for m, qt in obitos_pr.take(5):
    print(f'{NOMES.get(m, m):25s} {qt}')

Total de óbitos no Paraná: 20
Foz do Iguaçu             4
Curitiba                  3
Piraquara                 1
Toledo                    1
Corbélia                  1


**Resposta.** **Foz do Iguaçu**, com **4 óbitos**, seguido de Curitiba (3). O Paraná tem 20 óbitos nesta amostra; os demais municípios têm 1 cada.

In [17]:
# 8. Quantidade de municipios no Paraná com casos positivos
mun_pr_pos = positivos_pr.map(municipio)\
                         .filter(lambda m: m is not None)\
                         .distinct()
qt_mun_pr_pos = mun_pr_pos.count()
print('Municípios do Paraná com casos positivos:', qt_mun_pr_pos)

Municípios do Paraná com casos positivos: 189


**Resposta.** **189 municípios do Paraná** têm ao menos um caso positivo na amostra.

In [18]:
# 9. Quantidade de municipios no Paraná sem casos positivos
mun_pr = dados.filter(parana)\
              .map(municipio)\
              .filter(lambda m: m is not None)\
              .distinct()
mun_pr_sem_pos = mun_pr.subtract(mun_pr_pos)
qt_mun_pr = mun_pr.count()
qt_mun_pr_sem_pos = mun_pr_sem_pos.count()
print('Municípios do Paraná presentes na base:', qt_mun_pr)
print('Municípios do Paraná sem casos positivos:', qt_mun_pr_sem_pos)
print('Paraná tem 399 municípios; sem nenhum caso positivo na amostra (presentes ou não):', 399 - qt_mun_pr_pos)
print('Exemplos:', sorted(NOMES[m] for m in mun_pr_sem_pos.take(10)))

Municípios do Paraná presentes na base: 360
Municípios do Paraná sem casos positivos: 171
Paraná tem 399 municípios; sem nenhum caso positivo na amostra (presentes ou não): 210


Exemplos: ['Cafeara', 'Curitibapr', 'Icaraíma', 'Jesuítas', 'Perobal', 'Porto Barreiro', 'Quinta do Sol', 'Santa Cruz de Monte Castelo', 'Santa Tereza do Oeste', 'Santo Antônio Do Paraíso']


**Resposta.** Dos 360 nomes de município do Paraná presentes na base, **171 não têm nenhum caso positivo**. Considerando os 399 municípios do estado (inclusive os que nem aparecem na amostra), são **210** sem caso positivo registrado. A lista inclui nomes inválidos, como "Curitibapr", que são erros de digitação da própria base.

In [19]:
# 10. Estado com a maior taxa de falecimento de mulheres
mulheres = dados.filter(mulher)
total_mulheres_uf = mulheres.map(lambda r: (campo(r, 'estado'), 1)).reduceByKey(lambda a, b: a + b)
obitos_mulheres_uf = mulheres.filter(obito).map(lambda r: (campo(r, 'estado'), 1)).reduceByKey(lambda a, b: a + b)

taxa_uf = total_mulheres_uf.leftOuterJoin(obitos_mulheres_uf)\
                           .map(lambda kv: (kv[0], kv[1][1] or 0, kv[1][0], (kv[1][1] or 0) / kv[1][0]))\
                           .sortBy(lambda t: -t[3])\
                           .collect()
print(f"{'estado':20s} {'óbitos':>7s} {'mulheres':>9s} {'taxa':>8s}")
for uf, ob, tot, tx in taxa_uf:
    print(f'{str(uf):20s} {ob:>7d} {tot:>9d} {tx:>8.3%}')

estado                óbitos  mulheres     taxa
BAHIA                    373     66816   0.558%
RIO DE JANEIRO           166     51583   0.322%
PARANÁ                     8      3650   0.219%
MINAS GERAIS              94     58956   0.159%
SANTA CATARINA            41     36036   0.114%
SÃO PAULO                205    255172   0.080%
RIO GRANDE DO SUL         26     52296   0.050%


**Resposta.** **Bahia**, com taxa de **0,558 %** (373 óbitos em 66.816 mulheres). Taxa = óbitos de mulheres / total de mulheres do estado na base. Em seguida vêm Rio de Janeiro (0,322 %) e Paraná (0,219 %); a menor é a do Rio Grande do Sul (0,050 %).

In [20]:
# 11. Menor idade de mulher positiva para covid
idades_fem_pos = positivos.filter(mulher).map(idade).filter(lambda i: i is not None).cache()
print('Menor idade (valor bruto da base):', idades_fem_pos.min())
print('Quantidade de mulheres positivas com idade 0:', idades_fem_pos.filter(lambda i: i == 0).count())

Menor idade (valor bruto da base): 0


Quantidade de mulheres positivas com idade 0: 1308


**Resposta.** A menor idade registrada é **0 ano** (1.308 mulheres positivas com idade 0). Pode ser bebê com menos de um ano ou idade não preenchida gravada como zero; a base não permite distinguir.

In [21]:
# 12. Maior idade de mulher positiva para covid
print('Maior idade (valor bruto da base):', idades_fem_pos.max())
print('Cinco maiores valores brutos:', idades_fem_pos.top(5))
# idades acima de 120 anos são erros de digitação; maior idade plausível:
print('Maior idade plausível (<= 120 anos):', idades_fem_pos.filter(lambda i: i <= 120).max())
print('Registros descartados por idade > 120:', idades_fem_pos.filter(lambda i: i > 120).count())

Maior idade (valor bruto da base): 179


Cinco maiores valores brutos: [179, 144, 136, 135, 134]


Maior idade plausível (<= 120 anos): 120


Registros descartados por idade > 120: 6


**Resposta.** O maior valor bruto é **179 anos**, claramente um erro de digitação (há 6 registros acima de 120). Descartando idades acima de 120 anos, a maior idade é **120 anos**, valor ainda suspeito, mas possível de ter sido digitado de propósito.

In [22]:
# 13. Quantidade de casos positivos, em granularidade diária, ao longo do tempo
pos_dia = positivos.map(lambda r: (data(r), 1))\
                   .filter(lambda kv: kv[0] is not None)\
                   .reduceByKey(lambda a, b: a + b)\
                   .sortByKey()\
                   .collect()
print('Dias com casos positivos:', len(pos_dia), '| de', pos_dia[0][0], 'a', pos_dia[-1][0])
pico = max(pos_dia, key=lambda kv: kv[1])
print('Pico diário:', pico[0], 'com', pico[1], 'casos')
print()
for d, qt in pos_dia:
    print(d.isoformat(), qt)

Dias com casos positivos: 192 | de 2020-01-01 a 2020-08-02
Pico diário: 2020-07-13 com 6610 casos

2020-01-01 4
2020-01-06 1
2020-01-15 1
2020-01-16 1
2020-01-20 2
2020-01-21 2
2020-01-22 3
2020-01-24 2
2020-01-29 1
2020-01-30 3
2020-01-31 1
2020-02-01 7
2020-02-02 8
2020-02-03 8
2020-02-05 1
2020-02-06 2
2020-02-07 1
2020-02-09 1
2020-02-11 2
2020-02-12 2
2020-02-13 4
2020-02-14 3
2020-02-15 9
2020-02-16 8
2020-02-17 7
2020-02-18 10
2020-02-19 6
2020-02-20 10
2020-02-21 3
2020-02-22 7
2020-02-23 2
2020-02-24 7
2020-02-25 5
2020-02-26 7
2020-02-27 3
2020-02-28 4
2020-02-29 1
2020-03-01 1
2020-03-02 14
2020-03-03 3
2020-03-04 12
2020-03-05 4
2020-03-06 5
2020-03-07 1
2020-03-08 2
2020-03-09 9
2020-03-10 22
2020-03-11 20
2020-03-12 21
2020-03-13 29
2020-03-14 31
2020-03-15 27
2020-03-16 78
2020-03-17 64
2020-03-18 93
2020-03-19 77
2020-03-20 79
2020-03-21 50
2020-03-22 43
2020-03-23 145
2020-03-24 146
2020-03-25 159
2020-03-26 164
2020-03-27 222
2020-03-28 127
2020-03-29 121
2020-03-30 3

**Resposta.** A série diária completa está impressa acima: **192 dias** com casos positivos, de 01/01/2020 a 02/08/2020. O crescimento ganha força a partir de meados de março e o **pico diário é 13/07/2020, com 6.610 casos**. Os valores caem nos fins de semana (menos testes) e nos últimos dias da série (resultados ainda não lançados).

In [23]:
# 14. Quantidade de casos positivos, em granularidade semanal, ao longo do tempo
def semana(d):
    ano, sem, _ = d.isocalendar()
    return f'{ano}-S{sem:02d}'

pos_semana = positivos.map(lambda r: data(r))\
                      .filter(lambda d: d is not None)\
                      .map(lambda d: (semana(d), 1))\
                      .reduceByKey(lambda a, b: a + b)\
                      .sortByKey()\
                      .collect()
for s, qt in pos_semana:
    print(s, f'{qt:>7d}', '#' * (qt // 1000))

2020-S01       4 
2020-S02       1 
2020-S03       2 
2020-S04       9 
2020-S05      20 
2020-S06      13 
2020-S07      28 
2020-S08      45 
2020-S09      28 
2020-S10      41 
2020-S11     159 
2020-S12     484 
2020-S13    1084 #
2020-S14    1873 #
2020-S15    2359 ##
2020-S16    3546 ###
2020-S17    4284 ####
2020-S18    8214 ########
2020-S19    7937 #######
2020-S20    9764 #########
2020-S21   12906 ############
2020-S22   14973 ##############
2020-S23   16222 ################
2020-S24   17751 #################
2020-S25   22546 ######################
2020-S26   25735 #########################
2020-S27   25765 #########################
2020-S28   29594 #############################
2020-S29   30588 ##############################
2020-S30   26844 ##########################
2020-S31   15029 ###############


**Resposta.** A série semanal (semana ISO) está acima: menos de 50 casos por semana até a semana 10 (início de março), crescimento contínuo a partir da semana 11 e **pico na semana 29 (13 a 19/07/2020), com 30.588 casos**. A queda das semanas 30 e 31 reflete o atraso no lançamento dos resultados, e não necessariamente uma queda real.

In [24]:
# 15. Quantidade de pacientes que faleceram (evolucaoCaso) separados pela idade
obitos_idade = dados.filter(obito)\
                    .map(lambda r: (idade(r), 1))\
                    .filter(lambda kv: kv[0] is not None)\
                    .reduceByKey(lambda a, b: a + b)\
                    .sortByKey()\
                    .collect()
print('Total de óbitos com idade informada:', sum(q for _, q in obitos_idade))
print(obitos_idade)

Total de óbitos com idade informada: 2119
[(0, 53), (1, 3), (2, 5), (3, 2), (4, 1), (6, 1), (7, 1), (9, 1), (10, 2), (13, 1), (16, 2), (17, 2), (18, 5), (19, 4), (20, 5), (21, 6), (22, 5), (23, 5), (24, 2), (25, 3), (26, 6), (27, 2), (28, 5), (29, 4), (30, 7), (31, 6), (32, 9), (33, 12), (34, 10), (35, 10), (36, 10), (37, 15), (38, 15), (39, 15), (40, 10), (41, 15), (42, 10), (43, 19), (44, 15), (45, 22), (46, 21), (47, 24), (48, 15), (49, 17), (50, 21), (51, 21), (52, 25), (53, 18), (54, 27), (55, 40), (56, 41), (57, 39), (58, 30), (59, 35), (60, 40), (61, 36), (62, 45), (63, 42), (64, 37), (65, 35), (66, 40), (67, 47), (68, 59), (69, 45), (70, 52), (71, 45), (72, 39), (73, 49), (74, 46), (75, 52), (76, 48), (77, 42), (78, 36), (79, 49), (80, 55), (81, 40), (82, 46), (83, 46), (84, 41), (85, 43), (86, 27), (87, 42), (88, 24), (89, 21), (90, 26), (91, 24), (92, 29), (93, 13), (94, 16), (95, 11), (96, 8), (97, 7), (98, 2), (99, 4), (100, 1), (101, 2), (102, 2), (103, 1), (104, 1), (105,

In [25]:
# mesma informação agrupada em faixas de 10 anos, para leitura
faixas = {}
for i, q in obitos_idade:
    f = min(i // 10 * 10, 100)
    faixas[f] = faixas.get(f, 0) + q
for f in sorted(faixas):
    rot = f'{f}+' if f == 100 else f'{f}-{f + 9}'
    print(f'{rot:>7s} {faixas[f]:>5d}')

    0-9    67
  10-19    16
  20-29    43
  30-39   109
  40-49   168
  50-59   297
  60-69   426
  70-79   458
  80-89   385
  90-99   140
   100+    10


**Resposta.** Contagem de óbitos por idade na lista acima (2.119 óbitos). Agrupando em faixas de 10 anos: 0 a 9 anos, 67; 10 a 19, 16; 20 a 29, 43; 30 a 39, 109; 40 a 49, 168; 50 a 59, 297; **60 a 69, 426; 70 a 79, 458**; 80 a 89, 385; 90 a 99, 140; 100 ou mais, 10. A idade isolada com mais óbitos é 68 anos (59). Chama atenção o grupo de **idade 0, com 53 óbitos**, que provavelmente mistura bebês e idades não preenchidas.

In [26]:
# 16. Taxa de óbito (evolucaoCaso) por idade
total_idade = dados.map(lambda r: (idade(r), 1)).filter(lambda kv: kv[0] is not None).reduceByKey(lambda a, b: a + b)
obitos_idade_rdd = sc.parallelize(obitos_idade)
taxa_idade = total_idade.leftOuterJoin(obitos_idade_rdd)\
                        .map(lambda kv: (kv[0], kv[1][1] or 0, kv[1][0], (kv[1][1] or 0) / kv[1][0]))\
                        .filter(lambda t: t[0] <= 120)\
                        .sortBy(lambda t: t[0])\
                        .collect()
for i, ob, tot, tx in taxa_idade:
    print(f'{i:>3d} anos  óbitos {ob:>4d} / pacientes {tot:>6d}  taxa {tx:7.3%}')

  0 anos  óbitos   53 / pacientes   5972  taxa  0.887%
  1 anos  óbitos    3 / pacientes   3609  taxa  0.083%
  2 anos  óbitos    5 / pacientes   2984  taxa  0.168%
  3 anos  óbitos    2 / pacientes   2684  taxa  0.075%
  4 anos  óbitos    1 / pacientes   2543  taxa  0.039%
  5 anos  óbitos    0 / pacientes   2539  taxa  0.000%
  6 anos  óbitos    1 / pacientes   2553  taxa  0.039%
  7 anos  óbitos    1 / pacientes   2539  taxa  0.039%
  8 anos  óbitos    0 / pacientes   2594  taxa  0.000%
  9 anos  óbitos    1 / pacientes   2633  taxa  0.038%
 10 anos  óbitos    2 / pacientes   2702  taxa  0.074%
 11 anos  óbitos    0 / pacientes   2640  taxa  0.000%
 12 anos  óbitos    0 / pacientes   2849  taxa  0.000%
 13 anos  óbitos    1 / pacientes   3009  taxa  0.033%
 14 anos  óbitos    0 / pacientes   3419  taxa  0.000%
 15 anos  óbitos    0 / pacientes   3821  taxa  0.000%
 16 anos  óbitos    2 / pacientes   4567  taxa  0.044%
 17 anos  óbitos    2 / pacientes   5375  taxa  0.037%
 18 anos  

In [27]:
# taxa por faixa etária de 10 anos (soma de óbitos / soma de pacientes da faixa)
ob_f, tot_f = {}, {}
for i, ob, tot, _ in taxa_idade:
    f = min(i // 10 * 10, 100)
    ob_f[f] = ob_f.get(f, 0) + ob
    tot_f[f] = tot_f.get(f, 0) + tot
for f in sorted(tot_f):
    rot = f'{f}+' if f == 100 else f'{f}-{f + 9}'
    print(f'{rot:>7s}  {ob_f[f]:>5d} / {tot_f[f]:>7d}  = {ob_f[f] / tot_f[f]:7.3%}')

    0-9     67 /   30650  =  0.219%
  10-19     16 /   46843  =  0.034%
  20-29     43 /  185819  =  0.023%
  30-39    109 /  263789  =  0.041%
  40-49    168 /  216183  =  0.078%
  50-59    297 /  136354  =  0.218%
  60-69    426 /   71267  =  0.598%
  70-79    458 /   31327  =  1.462%
  80-89    385 /   13839  =  2.782%
  90-99    140 /    3522  =  3.975%
   100+     10 /     295  =  3.390%


In [28]:
# idades com ao menos 100 pacientes e maior taxa de óbito (evita taxas instáveis de idades raras)
for i, ob, tot, tx in sorted([t for t in taxa_idade if t[2] >= 100], key=lambda t: -t[3])[:5]:
    print(f'{i} anos: {ob}/{tot} = {tx:.2%}')

92 anos: 29/538 = 5.39%
97 anos: 7/148 = 4.73%
94 anos: 16/366 = 4.37%
95 anos: 11/253 = 4.35%
87 anos: 42/981 = 4.28%


**Resposta.** Taxa de óbito = óbitos / pacientes de cada idade (tabela por idade acima). A taxa cresce com a idade: por faixa de 10 anos, fica abaixo de 0,1 % de 10 a 49 anos, chega a 0,6 % na faixa de 60 a 69, **1,5 % na de 70 a 79, 2,8 % na de 80 a 89 e 4,0 % na de 90 a 99 anos**. Entre idades com pelo menos 100 pacientes, a maior taxa é a de 92 anos (5,4 %). A exceção é a idade 0 (0,9 %), coerente com a suspeita de idade não preenchida.

In [29]:
# 17. Idade média das mulheres positivas para covid
validas = idades_fem_pos.filter(lambda i: i <= 120)
soma, n = validas.map(lambda i: (i, 1)).reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))
print(f'Idade média das mulheres positivas: {soma / n:.2f} anos (n = {n})')
print(f'Média sem filtrar idades > 120: {idades_fem_pos.mean():.2f} anos')

Idade média das mulheres positivas: 40.89 anos (n = 142674)


Média sem filtrar idades > 120: 40.89 anos


**Resposta.** A idade média das mulheres positivas é **40,9 anos** (40,89; 142.674 mulheres com idade válida). Descartar as 6 idades acima de 120 anos não muda a média na segunda casa decimal.

In [30]:
# 18. Município do Paraná com a maior quantidade de mulheres positivos para covid
fem_pos_pr = positivos_pr.filter(mulher)\
                         .map(lambda r: (municipio(r), 1))\
                         .reduceByKey(lambda a, b: a + b)\
                         .sortBy(lambda kv: -kv[1])\
                         .take(5)
for m, qt in fem_pos_pr:
    print(f'{NOMES.get(m, m):25s} {qt}')

Curitiba                  189
São José dos Pinhais      28
Londrina                  23
Foz do Iguaçu             22
Cianorte                  21


**Resposta.** **Curitiba**, com **189 mulheres positivas**, seguida de São José dos Pinhais (27), Londrina (23), Foz do Iguaçu (22) e Cianorte (21).

In [31]:
# 19. Dia da semana com a maior quantidade de testes realizados
testes_dia_semana = dados.map(lambda r: data(r))\
                         .filter(lambda d: d is not None)\
                         .map(lambda d: (d.weekday(), 1))\
                         .reduceByKey(lambda a, b: a + b)\
                         .sortByKey()\
                         .collect()
for d, qt in testes_dia_semana:
    print(f'{DIAS[d]:8s} {qt:>8d}')

Segunda    190979
Terça      187798
Quarta     177292
Quinta     168976
Sexta      168193
Sábado      66826
Domingo     39936


**Resposta.** **Segunda-feira**, com **190.979 testes**. A quantidade cai ao longo da semana (terça 187.798, quarta 177.292, quinta 168.976, sexta 168.193) e despenca no sábado (66.826) e no domingo (39.936).

In [32]:
# 20. Dia da semana com maior quantidade de pacientes positivos para corona virus
pos_dia_semana = positivos.map(lambda r: data(r))\
                          .filter(lambda d: d is not None)\
                          .map(lambda d: (d.weekday(), 1))\
                          .reduceByKey(lambda a, b: a + b)\
                          .sortByKey()\
                          .collect()
for d, qt in pos_dia_semana:
    print(f'{DIAS[d]:8s} {qt:>8d}')

Segunda     54566
Terça       50969
Quarta      47312
Quinta      44921
Sexta       45759
Sábado      20858
Domingo     13463


**Resposta.** **Segunda-feira**, com **54.566 pacientes positivos**. O padrão acompanha o número de testes do item 19: os dias com mais testes são os com mais positivos.

In [33]:
# 21. Municipio, com mais de 500 testes, com a maior taxa de exames que não foram positivos para COVID (casos que foram testados porém não tiveram o resultadoTeste como Positivo)
# chave (município, estado) evita somar municípios homônimos de estados diferentes
testes_mun = dados.map(lambda r: ((municipio(r), campo(r, 'estado')), (1, 0 if positivo(r) else 1)))\
                  .filter(lambda kv: kv[0][0] is not None)\
                  .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))\
                  .filter(lambda kv: kv[1][0] > 500)\
                  .map(lambda kv: (NOMES[kv[0][0]], kv[0][1], kv[1][0], kv[1][1], kv[1][1] / kv[1][0]))\
                  .sortBy(lambda t: -t[4])
print('Municípios com mais de 500 testes:', testes_mun.count())
print(f"{'município':26s} {'estado':18s} {'testes':>7s} {'não pos.':>8s} {'taxa':>7s}")
for m, uf, tot, neg, tx in testes_mun.take(10):
    print(f'{m:26s} {uf:18s} {tot:>7d} {neg:>8d} {tx:>7.2%}')

Municípios com mais de 500 testes: 292
município                  estado              testes não pos.    taxa
Três Barras                SANTA CATARINA        1321     1272  96.29%
Congonhas                  MINAS GERAIS          1427     1352  94.74%
Viçosa                     MINAS GERAIS           570      536  94.04%
Vacaria                    RIO GRANDE DO SUL     1092     1025  93.86%
Serro                      MINAS GERAIS           570      532  93.33%
Guanambi                   BAHIA                  707      659  93.21%
Conceição do Mato Dentro   MINAS GERAIS          3375     3144  93.16%
Sant'Ana do Livramento     RIO GRANDE DO SUL      623      577  92.62%
Londrina                   PARANÁ                 539      499  92.58%
Uruguaiana                 RIO GRANDE DO SUL      588      539  91.67%


**Resposta.** Entre os 291 municípios com mais de 500 testes, a maior taxa de exames não positivos é de **Três Barras (SC): 96,29 %** (1.272 não positivos em 1.321 testes). Em seguida vêm Congonhas (MG, 94,74 %) e Viçosa (MG, 94,04 %). Do Paraná, o primeiro da lista é Londrina, em 9º lugar (92,58 %). A chave usada foi município e estado, para não somar municípios homônimos de estados diferentes.